In [1]:
import numpy as np
import torch
import gymnasium as gym
from dataclasses import dataclass


class CustomMountainCar(gym.Wrapper): # custom wrapper to not use the internal reward
    def __init__(self, env, goal_reward=1.0, step_reward=0.0):
        super().__init__(env)
        self.goal_reward = goal_reward
        self.step_reward = step_reward

    def step(self, action):
        obs, _, terminated, truncated, info = self.env.step(action)
        position, velocity = obs
        goal_position = self.unwrapped.goal_position
        reward = self.goal_reward if position >= goal_position else self.step_reward
        return obs, reward, terminated, truncated, info


@dataclass # typed container for batches of replay data
class ReplayBatch:
    obs: torch.Tensor
    actions: torch.Tensor
    rewards: torch.Tensor
    next_obs: torch.Tensor
    terminated: torch.Tensor
    truncated: torch.Tensor
    episode_id: torch.Tensor
    timestep: torch.Tensor
    indices: torch.Tensor

    def _len_(self):
        return self.obs.shape[0]


class TrajectoryReplayBuffer:
    def __init__(self, capacity, obs_dim, action_dim, device="cpu"):
        self.capacity = capacity
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.device = device

        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros((capacity, action_dim), dtype=np.float32)
        self.rewards = np.zeros((capacity, 1), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.terminated = np.zeros((capacity, 1), dtype=np.float32)
        self.truncated = np.zeros((capacity, 1), dtype=np.float32)

        self.episode_id = np.full((capacity,), -1, dtype=np.int64)
        self.timestep = np.full((capacity,), -1, dtype=np.int64)

        self.pos = 0
        self.size = 0
        self.full = False

        self.current_episode_id = 0
        self.episode_to_indices = {}

    def __len__(self):
        return self.size

    def add_episode(self, episode):
        ep_id = self.current_episode_id
        self.current_episode_id += 1

        ep_indices = []

        T = len(episode["obs"])
        for t in range(T):
            idx = self.pos

            if self.full:
                old_ep = self.episode_id[idx]
                if old_ep in self.episode_to_indices:
                    try:
                        self.episode_to_indices[old_ep].remove(idx)
                        if len(self.episode_to_indices[old_ep]) == 0:
                            del self.episode_to_indices[old_ep]
                    except ValueError:
                        pass

            self.obs[idx] = np.asarray(episode["obs"][t], dtype=np.float32)
            self.actions[idx] = np.asarray(episode["actions"][t], dtype=np.float32).reshape(-1)
            self.rewards[idx] = np.asarray([episode["rewards"][t]], dtype=np.float32)
            self.next_obs[idx] = np.asarray(episode["next_obs"][t], dtype=np.float32)
            self.terminated[idx] = np.asarray([episode["terminated"][t]], dtype=np.float32)
            self.truncated[idx] = np.asarray([episode["truncated"][t]], dtype=np.float32)

            self.episode_id[idx] = ep_id
            self.timestep[idx] = t

            ep_indices.append(idx)

            self.pos = (self.pos + 1) % self.capacity
            if self.size < self.capacity:
                self.size += 1
            else:
                self.full = True

        self.episode_to_indices[ep_id] = ep_indices

    def sample(self, batch_size):
        assert self.size > 0, "Buffer is empty"
        idxs = np.random.randint(0, self.size, size=batch_size)

        return ReplayBatch(
            obs=torch.tensor(self.obs[idxs], device=self.device),
            actions=torch.tensor(self.actions[idxs], device=self.device),
            rewards=torch.tensor(self.rewards[idxs], device=self.device),
            next_obs=torch.tensor(self.next_obs[idxs], device=self.device),
            terminated=torch.tensor(self.terminated[idxs], device=self.device),
            truncated=torch.tensor(self.truncated[idxs], device=self.device),
            episode_id=torch.tensor(self.episode_id[idxs], device=self.device),
            timestep=torch.tensor(self.timestep[idxs], device=self.device),
            indices=torch.tensor(idxs, device=self.device),
        )

    def sample_future_goal_batch(self, batch_size, min_k=1, max_k=None, gamma=0.99): # gamma controls the geometric distribution for positive-goal sampling
        assert self.size > 0, "Buffer is empty"

        valid_indices = []
        future_goal_indices = []

        tries = 0
        max_tries = batch_size * 20

        while len(valid_indices) < batch_size and tries < max_tries:
            idx = np.random.randint(0, self.size)
            ep_id = self.episode_id[idx]
            t = self.timestep[idx]

            if ep_id == -1 or ep_id not in self.episode_to_indices:
                tries += 1
                continue

            ep_idxs = self.episode_to_indices[ep_id]
            ep_len = len(ep_idxs)

            if t >= ep_len - 1:
                tries += 1
                continue

            max_valid_k = ep_len - 1 - t
            if max_k is not None:
                max_valid_k = min(max_valid_k, max_k)

            if max_valid_k < min_k:
                tries += 1
                continue

            geom_k = int(np.random.geometric(p=1.0 - gamma))  # k ~ Geometric(1-γ), matches sample_positive_future_goal
            k = max(min_k, min(geom_k, max_valid_k))  # clamp to [min_k, max_valid_k]
            future_t = t + k
            future_idx = ep_idxs[future_t]

            valid_indices.append(idx)
            future_goal_indices.append(future_idx)
            tries += 1

        assert len(valid_indices) > 0, "Could not sample valid future-goal pairs"

        idxs = np.array(valid_indices, dtype=np.int64)
        g_idxs = np.array(future_goal_indices, dtype=np.int64)

        batch = {
            "obs": torch.tensor(self.obs[idxs], device=self.device),
            "actions": torch.tensor(self.actions[idxs], device=self.device),
            "next_obs": torch.tensor(self.next_obs[idxs], device=self.device),
            "goals": torch.tensor(self.obs[g_idxs], device=self.device),
            "rewards": torch.tensor(self.rewards[idxs], device=self.device),
            "terminated": torch.tensor(self.terminated[idxs], device=self.device),
            "truncated": torch.tensor(self.truncated[idxs], device=self.device),
            "episode_id": torch.tensor(self.episode_id[idxs], device=self.device),
            "timestep": torch.tensor(self.timestep[idxs], device=self.device),
            "future_timestep": torch.tensor(self.timestep[g_idxs], device=self.device),
            "indices": torch.tensor(idxs, device=self.device),
            "goal_indices": torch.tensor(g_idxs, device=self.device),
        }
        return batch

    def sample_negative_future_goals(self, batch_size):
        idxs = np.random.randint(0, self.size, size=batch_size) # pick random indices from the buffer and indexing the observations
        return torch.tensor(self.obs[idxs], device=self.device)
    
    def sample_positive_future_goal(self, episode_index, timestep, k, gamma = 0.99):

        if episode_index not in self.episode_to_indices:
            raise ValueError(f"Episode index {episode_index} not found in buffer")

        ep_idxs = self.episode_to_indices[episode_index]
        ep_len = len(ep_idxs)

        if timestep >= ep_len - 1:
            raise ValueError(f"Timestep {timestep} is out of bounds for episode of length {ep_len}")

        max_valid_k = ep_len - 1 - timestep
        if k > max_valid_k:
            raise ValueError(f"k={k} is too large for episode of length {ep_len} at timestep {timestep}")

        geometric = torch.distributions.Geometric(probs=torch.tensor(1-gamma)) # geometric distribution to sample k with probability proportional to gamma^k
        steps_ahead = int(geometric.sample().item()) + 1 # sample k, cast to int, and add 1 to ensure it's at least t+1
        steps_ahead = min(steps_ahead, max_valid_k) # clamp so we never index past the end of the episode
        print(f"Sampled k={steps_ahead} from geometric distribution with gamma={gamma}")
        future_t = timestep + steps_ahead
        future_idx = ep_idxs[future_t]
        print(self.obs[future_idx])
        return torch.tensor(self.obs[future_idx], device=self.device) # return the future state as the positive goal

    def stats(self):
        return {
            "size": self.size,
            "capacity": self.capacity,
            "num_episodes": len(self.episode_to_indices),
            "current_episode_id": self.current_episode_id,
        }

### 2. Cell to Generate Data and push to replay buffer

In [ ]:
EPISODES = 100
MAX_HORIZON = 200
BUFFER_CAPACITY = 500000
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

env = gym.make(
    "MountainCarContinuous-v0",
    max_episode_steps=MAX_HORIZON,
)
env = CustomMountainCar(env)

obs_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

replay_buffer = TrajectoryReplayBuffer(
    capacity=BUFFER_CAPACITY,
    obs_dim=obs_dim,
    action_dim=action_dim,
    device=DEVICE,
)
print(DEVICE)
episodes = []

for epi in range(EPISODES):
    obs, _ = env.reset()
    done = False
    t = 0

    ep = {k: [] for k in ["obs", "actions", "rewards", "next_obs", "terminated", "truncated"]}

    while not done:
        action = env.action_space.sample().astype(np.float32)
        next_obs, reward, term, trunc, info = env.step(action)

        ep["obs"].append(obs.astype(np.float32))
        ep["actions"].append(action.astype(np.float32))
        ep["rewards"].append(np.float32(reward))
        ep["next_obs"].append(next_obs.astype(np.float32))
        ep["terminated"].append(np.float32(term))
        ep["truncated"].append(np.float32(trunc))

        obs = next_obs
        done = term or trunc
        t += 1

    ep_np = {k: np.asarray(v, dtype=np.float32) for k, v in ep.items()}
    episodes.append(ep_np)
    replay_buffer.add_episode(ep_np)

env.close()

print("Collected episodes:", len(episodes))
print("Replay stats:", replay_buffer.stats())

### 3. To check the distribution sampling for Contrastive loss

In [ ]:
batch = replay_buffer.sample(batch_size=256) # sampling state and actions randomly from the replay buffer
s = batch.obs
a = batch.actions

episode_idx = batch.episode_id # need this for actual future state sampling
timestep = batch.timestep # need this for actual future state sampling
s_pos_next = replay_buffer.sample_positive_future_goal(episode_idx[0].item(), timestep[0].item(), k=10) # sample a positive future goal index for the first sample in the batch, using k=10 for the geometric distribution
s_neg_next = replay_buffer.sample_negative_future_goals(batch_size=256) # sample negative future goals randomly from the buffer

positive_dot_product = torch.dot(s[0], s_pos_next)  # raw (unlearned) dot product, just to show data flow before representations are trained
print("positive dot product (raw):", positive_dot_product.item())

for s,a,s_next in zip(batch.obs, batch.actions, batch.next_obs):
    print("s:", s.cpu().numpy(), "a:", a.cpu().numpy(), "s_next:", s_next.cpu().numpy())

# the experience tuples are stored in the replay buffer as tensors, and to visualise it use cpu.numpy

### 4. The implementation of A Single Goal is All you Need.

In [5]:
# Now here is the NN to train the sampled batch data from the replay buffer, the goal is to learn a dynamics model that predicts s_next from (s,a) pairs.

import torch.nn as nn
import torch.optim as optim 
import torch.nn.functional as F

class StateActionRepresentationModel(nn.Module): # a simple MLP that takes in (s,a) and predicts s_next, use hidden_layers to control the depth of the MLP, and hidden_dim to control the width of the MLP

    def __init__(self, obs_dim, action_dim, hidden_dim=256, hidden_layers=5,normalise=False): # A single goal paper specifies no normalisation
        super().__init__()
        self.fc1 = nn.Linear(obs_dim + action_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers - 2)])
        self.fc_out = nn.Linear(hidden_dim, obs_dim)
        self.normalise = normalise

    def forward(self, x):
        x = F.relu(self.fc1(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        x = self.fc_out(x)
        if self.normalise: # if normalise is True, then we normalise the output to have unit norm, this can help with training stability and convergence, especially when using MSE loss, as it prevents the model from producing arbitrarily large outputs
            x = F.normalize(x, dim=-1)
        return x


In [6]:
class GoalRepresentationModel(nn.Module): # symmetric counterpart to StateActionRepresentationModel — maps a future/goal state sf to an embedding ψ(sf)

    def __init__(self, obs_dim, hidden_dim=256, hidden_layers=5, normalise=False): # same architecture choices as phi to keep the embedding space consistent
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers - 2)])
        self.fc_out = nn.Linear(hidden_dim, obs_dim) # output in the same embedding space as StateActionRepresentationModel so that the dot product phi(s,a)^T psi(sf) is well-defined
        self.normalise = normalise

    def forward(self, x):
        x = F.relu(self.fc1(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        x = self.fc_out(x)
        if self.normalise: # paper specifies no normalisation, but we keep the flag for experimentation
            x = F.normalize(x, dim=-1)
        return x


In [7]:
def contrastive_loss(phi_sa, psi_sf_pos, psi_sf_neg, reg_coef=0.01):
    # Implements the contrastive RL objective from Eq. 3 of the paper:
    #   max  E[ log( e^{phi(s,a)^T psi(sf+)} / (e^{phi(s,a)^T psi(sf+)} + sum_j e^{phi(s,a)^T psi(sf_j-)}) )
    #           - 0.01 * log( sum_j e^{phi(s,a)^T psi(sf_j)} )^2 ]   <- LogSumExp regularisation
    #
    # phi_sa     : (B, d)  state-action embeddings phi(s, a)
    # psi_sf_pos : (B, d)  positive future-state embeddings psi(sf+), sampled via Geometric(1-γ) k steps ahead
    # psi_sf_neg : (B, d)  negative future-state embeddings psi(sf-), sampled uniformly from the replay buffer (marginal distribution)
    #
    # For each anchor phi_sa[i]:
    #   - one positive  : phi_sa[i]^T psi_sf_pos[i]   (col 0 of all_logits)
    #   - B negatives   : phi_sa[i]^T psi_sf_neg[j] for all j  (cols 1..B)
    # This keeps positives and negatives cleanly separated (no accidental positive-as-negative contamination).

    B = phi_sa.shape[0]

    pos_logits = (phi_sa * psi_sf_pos).sum(dim=-1, keepdim=True)  # (B, 1)  one dot-product per positive pair
    neg_logits = phi_sa @ psi_sf_neg.T                            # (B, B)  phi(s_i,a_i)^T psi(sf_j-) for all j

    all_logits = torch.cat([pos_logits, neg_logits], dim=1)  # (B, 1+B): column 0 is the positive
    labels = torch.zeros(B, dtype=torch.long, device=phi_sa.device)  # label 0 = column 0 = positive

    infonce_loss = F.cross_entropy(all_logits, labels)  # minimising this maximises the infoNCE term

    logsumexp = torch.logsumexp(all_logits, dim=-1)  # (B,), log sum_j exp(logit_j)
    reg_loss  = reg_coef * logsumexp.pow(2).mean()   # penalise large LogSumExp values (matches -0.01 * (...) in the maximisation objective)

    return infonce_loss + reg_loss  # minimise this to maximise the paper's contrastive objective


In [8]:
class GoalConditionedActor(nn.Module): # goal-conditioned policy pi(a | s, g) — takes (s, g) and outputs a Gaussian action distribution

    def __init__(self, obs_dim, action_dim, hidden_dim=256, hidden_layers=5):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim * 2, hidden_dim)  # concatenate state s and goal g as input
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers - 2)])
        self.fc_mean = nn.Linear(hidden_dim, action_dim)  # output the mean of the action distribution
        self.log_std = nn.Parameter(torch.zeros(action_dim))  # learnable log standard deviation shared across the batch

    def forward(self, obs, goal):
        x = torch.cat([obs, goal], dim=-1)  # (B, obs_dim * 2)
        x = F.relu(self.fc1(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        mean = self.fc_mean(x)  # (B, action_dim)
        std = self.log_std.exp().expand_as(mean)  # (B, action_dim)
        return mean, std

    def sample_action(self, obs, goal):
        mean, std = self.forward(obs, goal)
        dist = torch.distributions.Normal(mean, std)
        action = dist.rsample()  # reparameterisation trick so gradients flow through the sampled action
        log_prob = dist.log_prob(action).sum(dim=-1, keepdim=True)  # (B, 1)
        entropy = dist.entropy().sum(dim=-1, keepdim=True)  # (B, 1), H(pi(.|s,g))
        return action, log_prob, entropy


In [11]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
TRAIN_STEPS  = 200     # number of outer iterations: each iteration = 1 trajectory + 1 gradient update (Alg. 1)
BATCH_SIZE   = 256      # samples per gradient update
LR           = 3e-4    # learning rate for all networks
ALPHA        = 0.1     # entropy coefficient for the actor loss (Eq. 4)
LOG_INTERVAL = 20      # print a summary every this many steps
EPISODES = 100
MAX_HORIZON = 500
BUFFER_CAPACITY = 500000
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# ── Single hard goal s* (Alg. 1, line 3) ──────────────────────────────────────
# For MountainCarContinuous the goal is to reach position >= 0.45 with any velocity
s_star = torch.tensor([0.45, 0.0], device=DEVICE, dtype=torch.float32) # the single target goal the agent is always commanded towards during data collection

# ── Re-open the environment for online data collection ─────────────────────────
# The environment was closed at the end of the initial data-collection cell above,
# so we open a fresh instance here for the training loop.
env_train = gym.make("MountainCarContinuous-v0", max_episode_steps=MAX_HORIZON)
env_train = CustomMountainCar(env_train)

obs_dim = env_train.observation_space.shape[0]
action_dim = env_train.action_space.shape[0]

replay_buffer = TrajectoryReplayBuffer(
    capacity=BUFFER_CAPACITY,
    obs_dim=obs_dim,
    action_dim=action_dim,
    device=DEVICE,
)
print(DEVICE)

phi_model = StateActionRepresentationModel(obs_dim, action_dim, hidden_dim=256, hidden_layers=5).to(DEVICE) # phi(s,a) encoder
psi_model = GoalRepresentationModel(obs_dim, hidden_dim=256, hidden_layers=5).to(DEVICE)                  # psi(sf) encoder
actor     = GoalConditionedActor(obs_dim, action_dim, hidden_dim=256, hidden_layers=5).to(DEVICE)         # pi(a|s,g)

critic_optimizer = optim.Adam(list(phi_model.parameters()) + list(psi_model.parameters()), lr=LR) # jointly optimise both representation networks
actor_optimizer  = optim.Adam(actor.parameters(), lr=LR)

# ── Training loop (Alg. 1, lines 2-4) ─────────────────────────────────────────
for step in range(TRAIN_STEPS):

    # ── Collect one trajectory using pi(a | s, g = s*) (Alg. 1, line 3) ─────
    obs_t, _ = env_train.reset()
    done = False
    ep = {k: [] for k in ["obs", "actions", "rewards", "next_obs", "terminated", "truncated"]}

    while not done:
        obs_tensor    = torch.tensor(obs_t, dtype=torch.float32, device=DEVICE).unsqueeze(0)  # (1, obs_dim)
        s_star_batch  = s_star.unsqueeze(0)  # (1, obs_dim) — always condition on the single hard goal s*

        with torch.no_grad():
            action, _, _ = actor.sample_action(obs_tensor, s_star_batch)  # sample a ~ pi(.|s, g=s*)
        action_np = action.squeeze(0).cpu().numpy()
        action_np = np.clip(action_np, env_train.action_space.low, env_train.action_space.high)  # keep action within valid bounds

        next_obs_t, reward, term, trunc, _ = env_train.step(action_np)

        ep["obs"].append(obs_t.astype(np.float32))
        ep["actions"].append(action_np.astype(np.float32))
        ep["rewards"].append(np.float32(reward))
        ep["next_obs"].append(next_obs_t.astype(np.float32))
        ep["terminated"].append(np.float32(term))
        ep["truncated"].append(np.float32(trunc))

        obs_t = next_obs_t
        done  = term or trunc

    replay_buffer.add_episode(ep)  # push the freshly collected trajectory into the buffer (Alg. 1, line 3)

    # ── Critic update: learn phi(s,a) and psi(sf) via contrastive loss (Eq. 3) ─
    batch = replay_buffer.sample_future_goal_batch(BATCH_SIZE) # sample (s, a, sf+) triples where sf+ is a future state reached from s

    obs     = batch["obs"]     # (B, obs_dim)
    actions = batch["actions"] # (B, action_dim)
    goals   = batch["goals"]   # (B, obs_dim)  positive future states sf+

    phi_sa   = phi_model(torch.cat([obs, actions], dim=-1))  # (B, d) state-action embedding
    psi_g    = psi_model(goals)                               # (B, d) positive future-state embedding (geometric k)

    neg_goals = replay_buffer.sample_negative_future_goals(BATCH_SIZE)  # (B, obs_dim) random states from marginal
    psi_neg   = psi_model(neg_goals)                                     # (B, d) negative future-state embeddings

    critic_loss = contrastive_loss(phi_sa, psi_g, psi_neg)

    critic_optimizer.zero_grad()
    critic_loss.backward()
    critic_optimizer.step()

    # ── Actor update: maximise phi(s,a)^T psi(g) + alpha * H(pi) (Eq. 4) ───────
    # The paper trains the actor with multiple goals from the buffer (multi-task),
    # even though data is collected by conditioning on the single hard goal s*.
    sampled_actions, _, entropy = actor.sample_action(obs, goals) # sample actions under the current policy, conditioned on the replay-buffer goals

    phi_sa_actor = phi_model(torch.cat([obs, sampled_actions], dim=-1)) # (B, d) re-compute embedding with freshly sampled actions
    psi_g_actor  = psi_model(goals)                                     # (B, d) goal embedding (no gradient through psi for actor update, consistent with actor-critic)

    q_values   = (phi_sa_actor * psi_g_actor.detach()).sum(dim=-1, keepdim=True) # (B, 1) dot product = log Q(s,a,g) - log rho(g)
    actor_loss = -(q_values + ALPHA * entropy).mean()                           # maximise Q-value + entropy bonus, hence the negative sign

    actor_optimizer.zero_grad()
    actor_loss.backward()
    actor_optimizer.step()

    if (step + 1) % LOG_INTERVAL == 0:
        ep_len = len(ep["obs"])
        print(f"Step {step+1:>4}/{TRAIN_STEPS} | ep_len: {ep_len:>3} | critic_loss: {critic_loss.item():.4f} | actor_loss: {actor_loss.item():.4f}")

env_train.close()
print("Training complete!")
print(f"Replay buffer size after training: {replay_buffer.stats()['size']}")


mps
Step   20/200 | ep_len: 500 | critic_loss: 5.7577 | actor_loss: 1.2468
Step   40/200 | ep_len: 500 | critic_loss: 5.5550 | actor_loss: 4.8610
Step   60/200 | ep_len: 500 | critic_loss: 5.5391 | actor_loss: 5.7620
Step   80/200 | ep_len: 500 | critic_loss: 5.5424 | actor_loss: 5.8262
Step  100/200 | ep_len: 500 | critic_loss: 5.5516 | actor_loss: 5.8398
Step  120/200 | ep_len: 500 | critic_loss: 5.5369 | actor_loss: 5.6969
Step  140/200 | ep_len: 500 | critic_loss: 5.4954 | actor_loss: 5.5902
Step  160/200 | ep_len: 500 | critic_loss: 5.5038 | actor_loss: 5.7844
Step  180/200 | ep_len: 500 | critic_loss: 5.4908 | actor_loss: 5.4273
Step  200/200 | ep_len: 500 | critic_loss: 5.5544 | actor_loss: 5.4474
Training complete!
Replay buffer size after training: 100000
